## Retrieval Augmented Generation with Langchain

Based on this [Langchain tutorial](https://docs.langchain.com/oss/python/langchain/rag)

#### 0. Installation of required libraries

In [ ]:
!pip install -U langchain langchain-text-splitters langchain-community bs4 langchain-huggingface transformers sentence-transformers faiss-cpu pypdf tiktoken

We will need a [HuggingFace token](https://huggingface.co/docs/hub/security-tokens) to access the models through the HuggingFace API. If you want to use models from other alternative providers you may need additional tokens for each provider. 

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token: ")


#### 1. Load documents

We can use [document loaders](https://docs.langchain.com/oss/python/integrations/document_loaders) to load data from different sources anc convert it into LangChain’s Document format. In this example we are using [PyPDF loader](https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader) to read pdf files. See the documentation for information on all the  options for loading different types of documents.



In this example we are reading a single pdf document. In the first cell we create one different langchain document per page of the original .pdf while in the second cell we create one single langchain document for the whole .pdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./data/study_guide.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print("Number of documents:",len(docs))
print(f"Beginning of the first document: {docs[0].page_content[:200]}\n")
print(f"Metadata of the first document: {docs[0].metadata}")

Number of documents: 5
Beginning of the first document: Teaching groups languages
You can view this information at the  of thisend
document.
Contact
ernest.valveny@uab.catEmail:
Ernest Valveny LlobetName:
2025/2026
Learning and Natural Language Processing


Metadata of the first document: {'producer': 'iText 2.0.8 (by lowagie.com)', 'creator': 'PyPDF', 'creationdate': '2025-09-12T03:44:25+02:00', 'title': 'Learning and Natural Language Processing', 'keywords': '', 'moddate': '2025-09-12T03:44:25+02:00', 'author': 'Valveny Llobet, Ernest', 'source': './data/study_guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [3]:
loader = PyPDFLoader(
    "./data/study_guide.pdf",
    mode="single",
)
docs = loader.load()

print("Number of documents:",len(docs))
print(f"Beginning of the first document: {docs[0].page_content[:200]}\n")
print(f"Metadata of the first document: {docs[0].metadata}")

Number of documents: 1
Beginning of the first document: Teaching groups languages
You can view this information at the  of thisend
document.
Contact
ernest.valveny@uab.catEmail:
Ernest Valveny LlobetName:
2025/2026
Learning and Natural Language Processing


Metadata of the first document: {'producer': 'iText 2.0.8 (by lowagie.com)', 'creator': 'PyPDF', 'creationdate': '2025-09-12T03:44:25+02:00', 'title': 'Learning and Natural Language Processing', 'keywords': '', 'moddate': '2025-09-12T03:44:25+02:00', 'author': 'Valveny Llobet, Ernest', 'source': './data/study_guide.pdf', 'total_pages': 5}


In this other example we load all .pdf files from a given folder using [GenericLoader](https://reference.langchain.com/python/langchain-community/document_loaders/generic/GenericLoader)

In [ ]:
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import PyPDFParser

loader = GenericLoader(
    blob_loader=FileSystemBlobLoader(
        path="./data/",
        glob="*.pdf",
    ),
    blob_parser=PyPDFParser(),
)
docs = loader.load()

print("Number of documents:",len(docs))
print(f"Beginning of the first document: {docs[0].page_content[:200]}\n")
print(f"Metadata of the first document: {docs[0].metadata}")

#### 2. Split the text into chunks

[Text splitters](https://docs.langchain.com/oss/python/integrations/splitters) allow to break documents into chunks. There are several available text splitters (see the documentation for detailed information).

In this example, in the first cell we are creating chunks based on the number of characters, with some degree of overlap. In the second cell, chunks are defined based on the number of tokens, using [tiktoken](https://github.com/openai/tiktoken), an implementation of BPE tokenization. 


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Initialize Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Create Documents (Chunks) From File
chunks = text_splitter.split_documents(docs)

print(len(chunks))
print(chunks[0].page_content)

17
Teaching groups languages
You can view this information at the  of thisend
document.
Contact
ernest.valveny@uab.catEmail:
Ernest Valveny LlobetName:
2025/2026
Learning and Natural Language Processing
Code: 106585
ECTS Credits: 6
Degree Type Year
Artificial Intelligence OT 3
Artificial Intelligence OT 4
Prerequisites
There are no official prerequisites but it is recommended to have completed the subjects of Fundamentals of
Programming I and II, Fundamentals of Mathematics I and II, Probability and Statistics, Data Engineering,
Fundamentals of Machine Learning, and Fundamentals of Natural Language.
Objectives and Contextualisation
This course provides an overview of the Natural Language Processing (NLP) applications, from classical
approaches for text processing to advanced methods for person-computer interaction. This course covers both
machine learning and deep learning techniques for NLP, considering both text and speech processing.
By the end of this course, students will be able 

In [6]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=10)

chunks = text_splitter.split_documents(docs)
print(len(chunks))
print(chunks[0].page_content)

36
Teaching groups languages
You can view this information at the  of thisend
document.
Contact
ernest.valveny@uab.catEmail:
Ernest Valveny LlobetName:
2025/2026
Learning and Natural Language Processing
Code: 106585
ECTS Credits: 6
Degree Type Year
Artificial Intelligence OT 3
Artificial Intelligence OT 4
Prerequisites
There are no official prerequisites but


#### 3. Embedding the chunks and storing the embeddings in a vector store

**Select the embedding model**

First, we need to select the embedding model to embed the chunks into a vector space. We can use any pre-trained model. In this example we use one of the available [embedding models in the HuggingFace Hub](https://huggingface.co/models?other=embeddings)

See the full [documentation on embedding models in LangChain](https://docs.langchain.com/oss/python/integrations/embeddings) for more information

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5783.56it/s]


**Select the vector store**

We also need to select a [vector store](https://docs.langchain.com/oss/python/integrations/vectorstores) to efficiently index and store all embedded chunks. In this example we are using FAISS as vector store but we can use any of the available options. 

In [8]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

# Create the vector store specifying the embedding method selected
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

**Embed and store document chunks**

Once we have selected an embedding and a vector store we can add all the chunks to the vector store. Chunks will be automatically embedded before adding them to the vector store. 

In [9]:
# Save Document Chunks to Vector Store
ids = vector_store.add_documents(chunks)

#### 4. Retrieval of relevant chunks to a given query

Just specify the query and use the method `similarity_search` from the vector store. It will embed the query and return the most similar chunks according to the configuration of the embedding and the vector store previously defined.

In [18]:
# Query the Vector Store
results = vector_store.similarity_search(
    'How many members can be in a project group?',
    k=5
)

# Print Resulting Chunks
for res in results:
    print(f"* {res.page_content} [{res.metadata}]\n\n")

*  of distribution of roles, work
planning, assignment of tasks, management of available resources, conflicts, etc. To develop the project, the
groups will work autonomously, while the practical sessions will be used (1) for the teacher to present the
project theme and discuss possible approaches, (2) for monitoring the status of the project and (3) for the
teams to present their final results.
The above activities will be complemented by a system of tutoring and [{'producer': 'iText 2.0.8 (by lowagie.com)', 'creator': 'PyPDF', 'creationdate': '2025-09-12T03:44:25+02:00', 'title': 'Learning and Natural Language Processing', 'keywords': '', 'moddate': '2025-09-12T03:44:25+02:00', 'author': 'Valveny Llobet, Ernest', 'source': './data/study_guide.pdf', 'total_pages': 5, 'page': 2, 'page_label': '3'}]


*  retake the Test. After the retakes, the maximum grade which can be obtained is 8.
Project Grade
The project carries an essential weight in the overall mark of the subject. Developing the

We can also return the similarity score in case we can filter chunks by similarity. 

In [20]:
# Also return the similarity score
# Note that providers implement different scores; the score here    
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score('What is the name of the lecturer of the course Learning and NLP?', k=5)
for doc, score in results:
    print(f"Score: {score}\n")
    print(f"Document: {doc}\n")

Score: 0.8822469711303711

Document: page_content=' schedule set by the centre or degree programme, 15 minutes of one class will be
reserved for students to evaluate their lecturers and their courses or modules through questionnaires.
Assessment
Continous Assessment Activities
Title Weighting Hours ECTS Learning Outcomes
Portfolio 49% 0 0 1, 9, 10, 2, 3, 5, 4, 6, 11, 12
Project 30% 2 0.08 1, 9, 10, 6, 7, 11,' metadata={'producer': 'iText 2.0.8 (by lowagie.com)', 'creator': 'PyPDF', 'creationdate': '2025-09-12T03:44:25+02:00', 'title': 'Learning and Natural Language Processing', 'keywords': '', 'moddate': '2025-09-12T03:44:25+02:00', 'author': 'Valveny Llobet, Ernest', 'source': './data/study_guide.pdf', 'total_pages': 5, 'page': 2, 'page_label': '3'}

Score: 1.004332184791565

Document: page_content=' of the team.
Content
Fundamentals of NLP
Semantic Analysis
Pragmatic Analysis
Recurrent Neural Networks for NLP Applications
Transformers for NLP Applications
Foundation Models for NLP Ap

#### 5. Answer generation

**Select the LLM model**

We will need to select the LLM model that will be used to generate the answer. You can use any model from any provider you may have access to. 

In this example we will use models from the HuggingFace Hub using [`ChatHuggingFace`](https://docs.langchain.com/oss/python/integrations/chat/huggingface).

There are two different ways to use LLM models:
- Loading the model locally (you need enough local computational resources to run the model)
- Accessing the model remotely on HuggingFace servers (you need an API token)

In [10]:
# LOCAL ACCESS TO THE MODEL

from langchain.chat_models import init_chat_model

model = init_chat_model(
    "microsoft/Phi-3-mini-4k-instruct",
    model_provider="huggingface",
    temperature=0.7,
    max_tokens=1024,
)

Loading weights: 100%|██████████| 195/195 [00:00<00:00, 5680.42it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [11]:
# REMOTE ACCESS TO THE MODEL

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro:fireworks-ai",
    temperature=0.7,
    max_new_tokens=1024,
)
model = ChatHuggingFace(llm=llm)

**Create the RAG chain to answer the question using the retrieved context**

In [12]:
# Create the function to format the retrieved documents before sending them to the model
# In this case, we just concatenate the retrieved documents, but more complex formatting can be done here.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
from langchain_core.prompts import PromptTemplate

# Create the Prompt Template
# In this case, the prompt instructs the model to use the provided context to answer the user's question, and to refrain from answer 
# if the context does not provide enough information to do so.
# The prompt includes placeholders for the retrieved context and the user's query, which will be filled in when the chain is executed.

prompt_template = """Use the context provided to answer 
the user's question below. If you do not know the answer 
based on the context provided, tell the user that you do 
not know the answer to their question based on the context
provided and that you are sorry.

context: {context}

question: {query}

answer: """

# Create Prompt Instance from template
custom_rag_prompt = PromptTemplate.from_template(prompt_template)

In [14]:
# Create the RAG Chain

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define the retriever to be used in the chain. In this case, we use the retriever created from the vector store. 
retriever = vector_store.as_retriever()

# Define the pipeline of the RAG chain. 
# First, the query is forwarded to the retriever, which retrieves the relevant documents, and the retrieved documents are formatted using the format_docs function. 
# This creates the context that will be provided to the model, along with the original query.
# Then, the context and query are passed to the custom prompt, which formats the input for the model, according to the template previously defined. 
# Finally, the prompt is forwarded to the model, that generates a response, which is parsed as a string using the StrOutputParser.
rag_chain = (
  {"context": retriever | format_docs, "query": RunnablePassthrough()}
  | custom_rag_prompt
  | model
  | StrOutputParser()
)

In [17]:
# Execute the RAG chain with a specific query. 
rag_chain.invoke("How many members can be in a project group?")

'Based on the provided context, a project group can have two or three students.'

In [15]:
# Execute the RAG chain with a specific query. 
rag_chain.invoke("What is the name of the lecturer of the course Learning and NLP?")

'I\'m sorry, but the provided context does not include the name of the lecturer for the course "Learning and NLP." The context only covers details about assessment, content, methodology, and scheduling, with no mention of any lecturer names.'

In [16]:
# Execute the RAG chain with a query that cannot be answered with the retrieved context.
rag_chain.invoke("What is the purpose of life?")

'I am sorry, but the provided context does not contain information about the purpose of life. The context focuses on competences and learning outcomes in Artificial Intelligence, such as problem-solving, knowledge representation, natural language processing, and data management. Therefore, I cannot answer your question based on this context.'